# "Ruse of Reuse" Hackathon Starter Kit

This notebook provides:
- Helper functions to load the hackathon task data;
- A reusable pipeline for evaluation of reuse detection methods;
- A **`participant_method()`** function for "plugging in" new approaches;
- A lightweight baseline reuse detection method relying on Bible verse embeddings (delivered in a ChromaDB vectorstore) and simple thresholding.


In [ ]:
team_name = 'Unknown'
method = 'embeddings'

## ▶️ Google Colab

**This section is intended to be run in a Google Colab environment. If you are running this notebook locally, skip the cells below and go to 'Evaluation Pipeline' section.**

Run this cell to configure environment, install all necessary packages, and  download the data.

If you are running this notebook locally, follow the instructions in the `README.md` to set up your environment and load the data.


In [1]:
%%capture
from google.colab import userdata
import os

try:
    os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    os.environ['GITHUB_TOKEN'] = ''
    print("Warning: GITHUB_TOKEN secret not found. Defaulting to empty string.")

try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except userdata.SecretNotFoundError:
    os.environ['OPENAI_API_KEY'] = ''
    print("Warning: OPENAI_API_KEY secret not found. Defaulting to empty string.")

!gh auth setup-git
!gh repo clone glsch/ruse_of_reuse


ModuleNotFoundError: No module named 'google.colab'

In [2]:
%%capture

!cp -a ./ruse_of_reuse/. ./
!rm -rf ./ruse_of_reuse


In [3]:
%%capture
!python -m pip install -e .


In [4]:
!python -m ruse_of_reuse download


2026-03-03 19:20:14,477 - ruse_of_reuse.utils - ERROR - No link provided!
2026-03-03 19:20:14,478 - ruse_of_reuse.utils - INFO - Checking for archive at /Users/glebschmidt/PycharmProjects/ruse_of_reuse/data/data.zip
2026-03-03 19:20:14,478 - ruse_of_reuse.utils - INFO - Archive not found. Downloading...
2026-03-03 19:20:14,478 - ruse_of_reuse.utils - ERROR - Failed to download file: Either url or id has to be specified
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/glebschmidt/PycharmProjects/ruse_of_reuse/src/ruse_of_reuse/__main__.py", line 88, in <module>
    cli()
  File "/Users/glebschmidt/PycharmProjects/ruse_of_reuse/src/ruse_of_reuse/__main__.py", line 84, in cli
    sys.exit(main())
             ^^^^^^
  File "/Users/glebschmidt/PycharmProjects/ruse_of_reuse/src/ruse_of_reuse/__main__.py", line 77, in main
    return args.func(args)
           ^^^^^^^^^^^^^^^
  File "/Us

Although the task data is delivered already pre-processed, you can produce it from raw data by running:


In [5]:
!python -m ruse_of_reuse preprocess


Scanning 53 XML file(s) in /Users/glebschmidt/PycharmProjects/ruse_of_reuse/data/raw …

Quote-source statistics:
- XML files scanned: 53
- Files with <quote source="..."> (before resolution): 53
- Total quotations with source attribute (before resolution): 752
- Files with quotations containing mapped Bible abbreviations (before resolution): 53
- Total quotations containing mapped Bible abbreviations (before resolution): 530

Preprocessing data into /Users/glebschmidt/PycharmProjects/ruse_of_reuse/data/task …

Computed statistics:
- Files: 53
- Spans: 561
- Individual verses: 705
- Biblical books: 47
- Total verses (True Positives): 824

Report written: /Users/glebschmidt/PycharmProjects/ruse_of_reuse/data/task/preprocess_report.json
Done.


## Evaluation Pipeline

1. Make sure to have the following file structure:
```
ruse_of_reuse/
├── data/                                # Should be downloaded
│   ├── raw/
│   ├── task/
│   ├── vectorstores/
│   ├── book_mapping.tsv
│   ├── reference_mapping.json
│   └── bible.tsv
├── notebooks/
│   └── method_evaluation.ipynb          # This notebook
└── src/                                 # Package code
    ├── ruse_of_reuse/
    └── ...
```
2. Run cells top-to-bottom once to load helper functions and the data.
3. To get baseline performance, run the rest of the notebook without modifying the code `participant_method()`  it to get .
3. To implement your own approach, modify the `participant_method()`. **It is important that the structure of output remains the same!**

For more information, see docstrings of the

### Expected Output Structure
Metric computation depends on `participant_method()` returning correctly structured output.

`participant_method()` must return: `List[Dict[str, Any]]`

Each dictionary represents **one predicted pair**. For example,
```
{
    "problem_id": "D.LEO-0458-541",
    "reference: "john_1:3"
}
```
`"reference"` key must contain only **one** biblical verse structured as follows:
``
<book_code>_<chapter>:<verse>
``
Book codes must be consistent with those used in `bible.tsv`. They can be found either in `bible.tsv` or, in a more concise form, in `book_mapping.tsv`.

More formally:
* `problem_id` (required, `str`)
  - Must be the filename stems from `data/task/problems/*.txt`.
  > Example: `"D.LEO-0458-541"`
* `reference` (required, `str`)
  - A single biblical verse reference in format: `<book_code>_<chapter>:<verse>`.
  **Span references, e.g., ``john_1:3-5`` must be expanded into three separate verses each represented by their own dictionary!**
  - Book code should be consistent with `bible.tsv`. Lowercase is applied by the evaluator.
  - Example: `"john_1:3"`

Dictionary may include any number of optional keys. For example, the proposed baseline method also outputs contains:
*`score` (optional, `float`)
  - Cosine similarity. Not used directly in scoring, but useful for debugging and analysis.
*`method` (optional, `str`)
  - Method name label, `"simple_embedding"`.

### Important Evaluation Behavior

- Scoring uses only `(problem_id, reference)` pairs.
- Duplicate predictions of the same pair are deduplicated automatically.
- Missing required keys (`problem_id`, `reference`) effectively drop a row from scoring.
- Extra keys are allowed but ignored by the scorer.

### Examples
#### Valid output
```python
[
  {"problem_id": "D.LEO-0458-541", "reference": "john_1:3"},
  {"problem_id": "D.LEO-0458-541", "reference": "john_1:4", "score": 0.81, "method": "baseline"},
]
```

#### Invalid output
```
# Problem identifier does not correspond to a filename stem from data/task/problems
[
  {
    "problem_id": "123",
    "reference": "john_1:3"
  },
  ...
]

# "reference" key is missing
[
  {
    "problem_id": "D.LEO-0458-541",
  },
  ...
]

# Reference to multiple verses
[
  {
    "problem_id": "D.LEO-0458-541",
    "reference": "john_1:3-5"
  },
  ...
]

# Invalid book code
[
  {
    "problem_id": "D.LEO-0458-541",
    "reference": "jhn_1:3"
  },
  ...
]
```


In [32]:
from __future__ import annotations

import json
import logging
import os
import re
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence

import pandas as pd
from tqdm.auto import tqdm

from ruse_of_reuse import setup_logging
from ruse_of_reuse.logging import ModuleLoggingConfig

setup_logging(
    ModuleLoggingConfig(
        default_level=logging.DEBUG,   # INFO or DEBUG
        log_file="method_evaluation.log",
    )
)

logger = logging.getLogger("method_evaluation")
logger.setLevel(logging.DEBUG)

# optional: reduce noise from libs
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("chromadb").setLevel(logging.INFO)

In [33]:
def find_project_root(start: Optional[Path] = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "task").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing data/task")


PROJECT_ROOT = find_project_root()
TASK_DIR = PROJECT_ROOT / "data" / "task"
PROBLEMS_DIR = TASK_DIR / "problems"
SOLUTIONS_DIR = TASK_DIR / "solutions"
CHROMA_DIR = PROJECT_ROOT / "data" / "vectorstores" / "chroma"


def normalize_reference(ref: str) -> str:
    return str(ref).strip().lower()


def load_task_problems(task_dir: Path = TASK_DIR) -> Dict[str, str]:
    problems_dir = Path(task_dir) / "problems"
    if not problems_dir.exists():
        raise FileNotFoundError(f"Missing problems directory: {problems_dir}")

    problems: Dict[str, str] = {}
    for path in sorted(problems_dir.glob("*.txt")):
        problems[path.stem] = path.read_text(encoding="utf-8")
    return problems


def load_task_ground_truth(task_dir: Path = TASK_DIR) -> Dict[str, List[str]]:
    solutions_dir = Path(task_dir) / "solutions"
    if not solutions_dir.exists():
        raise FileNotFoundError(f"Missing solutions directory: {solutions_dir}")

    out: Dict[str, List[str]] = {}
    for path in sorted(solutions_dir.glob("*.json")):
        payload = json.loads(path.read_text(encoding="utf-8"))
        refs = set()
        for item in payload:
            for ref in item.get("resolved_references", []):
                refs.add(normalize_reference(ref))
        out[path.stem] = sorted(refs)
    return out


def load_bible_tsv(project_root: Path = PROJECT_ROOT) -> pd.DataFrame:
    candidates = [
        project_root / "data" / "raw" / "bible.tsv",
        project_root / "data" / "bible.tsv",
        project_root / "data" / "hackathon_dataset" / "bible.tsv",
    ]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        raise FileNotFoundError(f"Could not locate bible.tsv in: {candidates}")

    df = pd.read_csv(path, sep="\t")
    df["reference"] = (
        df["book_code"].astype(str).str.strip().str.lower()
        + "_"
        + df["chapter_number"].astype(str)
        + ":"
        + df["verse_index"].astype(str)
    )
    return df


problems = load_task_problems(TASK_DIR)
ground_truth = load_task_ground_truth(TASK_DIR)
print(f"Loaded problems: {len(problems)}")
print(f"Loaded ground-truth docs: {len(ground_truth)}")


Loaded problems: 53
Loaded ground-truth docs: 53


### Evaluation API

- `run_method_on_dataset()` is method-agnostic and should be reused for any future approach.
- `score_predictions()` computes precision/recall/F1 over a list of `(problem_id, reference)` pairs provided as dictionaries. Automatically deduplicated during scoring is applied in case of multiple identical problem-verse pairs.


In [ ]:
def flatten_truth_pairs(ground_truth_by_problem: Dict[str, Sequence[str]]) -> set[tuple[str, str]]:
    pairs = set()
    for problem_id, refs in ground_truth_by_problem.items():
        for ref in refs:
            pairs.add((problem_id, normalize_reference(ref)))
    return pairs


def flatten_prediction_pairs(predictions: Sequence[Dict[str, Any]]) -> set[tuple[str, str]]:
    pairs = set()
    for row in predictions:
        pid = str(row.get("problem_id", "")).strip()
        ref = normalize_reference(row.get("reference", ""))
        if pid and ref:
            pairs.add((pid, ref))
    return pairs


def score_predictions(
    predictions: Sequence[Dict[str, Any]],
    ground_truth_by_problem: Dict[str, Sequence[str]],
) -> Dict[str, float]:
    pred_pairs = flatten_prediction_pairs(predictions)
    true_pairs = flatten_truth_pairs(ground_truth_by_problem)

    tp = len(pred_pairs & true_pairs)
    fp = len(pred_pairs - true_pairs)
    fn = len(true_pairs - pred_pairs)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    result = {
        "true_positives": float(tp),
        "false_positives": float(fp),
        "false_negatives": float(fn),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "team_name": team_name,
        "method": method,
    }

    try:
        import urllib.request
        import json as _json

        payload = _json.dumps(result).encode("utf-8")
        req = urllib.request.Request(
            url="https://ruseofreuse.humanities.tools/api/leaderboard",
            data=payload,
            headers={"Content-Type": "application/json"},
            method="POST",
        )
        with urllib.request.urlopen(req, timeout=10) as resp:
            logger.info("Leaderboard submission status: %s", resp.status)
    except Exception as exc:
        logger.warning("Leaderboard submission failed: %s", exc)

    return result


MethodFn = Callable[[str, str, Dict[str, Any]], List[Dict[str, Any]]]

def run_method_on_dataset(
    method_fn: MethodFn,
    problems_by_id: Dict[str, str],
    method_context: Dict[str, Any],
    *,
    problem_ids: Optional[Sequence[str]] = None,
    max_problems: Optional[int] = None,
    show_progress: bool = True,
) -> List[Dict[str, Any]]:
    selected = list(problem_ids) if problem_ids is not None else sorted(problems_by_id.keys())
    if max_problems is not None:
        selected = selected[:max_problems]

    iterator: Iterable[str] = selected
    if show_progress:
        iterator = tqdm(selected, desc="Running method", unit="problem")

    out: List[Dict[str, Any]] = []
    for problem_id in iterator:
        text = problems_by_id[problem_id]
        rows = method_fn(problem_id, text, method_context)
        out.extend(rows)
    return out


### Naive Embedding Baseline

This baseline is intentionally lightweight:
- split problem text into chunks (sentence / sentence-window / char windows)
- embed each chunk with the selected model
- retrieve top-k Bible verses from Chroma
- accept references with `similarity >= similarity_threshold`

Tune first:
- `mode`, `sentences_per_chunk`, `sentence_stride`
- `top_k`
- `similarity_threshold`


In [35]:
################################################################
## Helpers used by simple_embedding_method()
################################################################
def _import_chromadb():
    import chromadb
    return chromadb


def _import_sentence_transformer():
    from sentence_transformers import SentenceTransformer
    return SentenceTransformer


def _import_openai_client():
    from openai import OpenAI
    return OpenAI


def _sanitize_component(value: str) -> str:
    return re.sub(r"[^a-zA-Z0-9]+", "_", str(value)).strip("_").lower() or "default"


def _guess_collection_name(collection_prefix: str, provider: str, model_name: str, max_len: int = 63) -> str:
    base = f"{_sanitize_component(collection_prefix)}__{_sanitize_component(provider)}__{_sanitize_component(model_name)}"
    if len(base) <= max_len:
        return base

    import hashlib

    digest = hashlib.sha1(base.encode("utf-8")).hexdigest()[:10]
    keep = max_len - len(digest) - 2
    trimmed = base[:keep].rstrip("_-")
    return f"{trimmed}__{digest}"


def _list_collection_names(client: Any) -> List[str]:
    out = []
    for item in client.list_collections():
        if isinstance(item, str):
            out.append(item)
        else:
            name = getattr(item, "name", None)
            if name:
                out.append(name)
    return out


def _resolve_collection_name(
    client: Any,
    *,
    provider: str,
    model_name: str,
    collection_prefix: str,
    chroma_dir: Path,
    openai_api_key: Optional[str] = None,
    openai_base_url: Optional[str] = None,
    openai_organization: Optional[str] = None,
) -> str:
    provider_norm = str(provider).strip().lower()
    model_norm = str(model_name).strip().lower()
    prefix_norm = str(collection_prefix).strip().lower()

    if provider_norm not in {"hf", "openai"}:
        raise ValueError("provider must be 'hf' or 'openai'")

    def _matches(metadata: Any) -> bool:
        if not isinstance(metadata, dict):
            return False
        p = str(metadata.get("provider", "")).strip().lower()
        m = str(metadata.get("model_name", "")).strip().lower()
        pref = str(metadata.get("collection_prefix", "")).strip().lower()
        return p == provider_norm and m == model_norm and pref == prefix_norm

    existing_names = sorted(set(_list_collection_names(client)))
    for name in existing_names:
        try:
            collection = client.get_collection(name)
        except Exception:
            continue
        if _matches(getattr(collection, "metadata", None)):
            return name

    from ruse_of_reuse.vector_store import build_biblical_vectorstores

    build_biblical_vectorstores(
        hf_models=[model_name] if provider_norm == "hf" else [],
        openai_models=[model_name] if provider_norm == "openai" else [],
        persist_directory=Path(chroma_dir),
        collection_prefix=prefix_norm,
        openai_api_key=openai_api_key,
        openai_base_url=openai_base_url,
        openai_organization=openai_organization,
        show_progress=True,
    )

    refreshed_names = sorted(set(_list_collection_names(client)))
    for name in refreshed_names:
        try:
            collection = client.get_collection(name)
        except Exception:
            continue
        if _matches(getattr(collection, "metadata", None)):
            return name

    raise ValueError(
        f"No biblical collection found for provider={provider_norm}, model={model_norm}, prefix={prefix_norm} even after build attempt."
    )

def _build_query_embedder(
    provider: str,
    model_name: str,
    *,
    hf_batch_size: int = 64,
    openai_batch_size: int = 128,
    openai_api_key: Optional[str] = None,
    openai_base_url: Optional[str] = None,
    openai_organization: Optional[str] = None,
    query_prompt: Optional[Dict[str, Any]] = None,
    **kwargs,
) -> Callable:
    """
    Responsible for setting up a function used for embedding of query texts.

    :param provider:
    :param model_name:
    :param hf_batch_size:
    :param openai_batch_size:
    :param openai_api_key:
    :param openai_base_url:
    :param openai_organization:
    :param query_prompt: Can be used to pass additional keyword arguments to :meth:`SentenceTransformer.encode`.
    :return:
    """
    provider = provider.lower().strip()

    if provider == "hf":

        SentenceTransformer = _import_sentence_transformer()
        model = SentenceTransformer(model_name, device=kwargs.get("device", "cpu"))

        def embed(texts: Sequence[str]) -> List[List[float]]:
            encode_kwargs = {
                "batch_size": max(1, int(hf_batch_size)),
                "show_progress_bar": False,
                "convert_to_numpy": True,
                "normalize_embeddings": False,
            }
            if query_prompt is not None and isinstance(query_prompt, dict):
                encode_kwargs["encode_kwargs"] = query_prompt

            vecs = model.encode(
                list(texts),
                **encode_kwargs
            )
            return vecs.tolist()

        return embed

    if provider == "openai":
        OpenAI = _import_openai_client()
        api_key = openai_api_key or os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("OpenAI provider requires OPENAI_API_KEY or openai_api_key.")
        client = OpenAI(api_key=api_key, base_url=openai_base_url, organization=openai_organization)

        def embed(texts: Sequence[str]) -> List[List[float]]:
            all_embeddings: List[List[float]] = []
            batch_size = max(1, int(openai_batch_size))
            for i in range(0, len(texts), batch_size):
                batch = list(texts[i:i + batch_size])
                response = client.embeddings.create(model=model_name, input=batch)
                data = sorted(response.data, key=lambda x: x.index)
                all_embeddings.extend([list(item.embedding) for item in data])
            return all_embeddings

        return embed

    raise ValueError("provider must be 'hf' or 'openai'")


def split_into_chunks(
    text: str,
    *,
    mode: str = "sentence",
    sentences_per_chunk: int = 2,
    sentence_stride: int = 1,
    char_chunk_size: int = 500,
    char_chunk_overlap: int = 100,
    min_chunk_chars: int = 30,
) -> List[str]:
    """
    Split a text into chunks using sentence- or character-based strategies.

    Supported modes:

    - "sentence": split on hard punctuation (.!?;:) followed by whitespace;
    - "sentence_window": sliding window over sentences;
    - "char"``: sliding window over character spans.

    Sentence splitting uses the regex ``(?<=[.!?;:])\\s+``.

    :param text:
    :param mode: "sentence", "sentence_window", "char"
    :param sentences_per_chunk: When mode="sentence_window", window size (in sentences)
    :param sentence_stride: When mode="sentence_window", step size (in sentences)
    :param char_chunk_size: When mode="char", window size (in characters)
    :param char_chunk_overlap: When mode="char", overlap (in characters).
    :param min_chunk_chars: Drop shorter chunks or not
    :returns: A list of chunks, each at least min_chunk_chars long.

    """

    text = str(text or "").strip()
    if not text:
        return []

    mode = mode.lower().strip()
    chunks: List[str] = []

    if mode in {"sentence", "sentence_window"}:
        sentences = [s.strip() for s in re.split(r"(?<=[.!?;:])\s+", text) if s.strip()]
        if not sentences:
            return [text]

        if mode == "sentence":
            chunks = sentences
        else:
            win = max(1, int(sentences_per_chunk))
            step = max(1, int(sentence_stride))
            for i in range(0, len(sentences), step):
                piece = " ".join(sentences[i:i + win]).strip()
                if piece:
                    chunks.append(piece)

    elif mode == "char":
        size = max(1, int(char_chunk_size))
        overlap = max(0, int(char_chunk_overlap))
        step = max(1, size - overlap)
        for i in range(0, len(text), step):
            piece = text[i:i + size].strip()
            if piece:
                chunks.append(piece)
    else:
        raise ValueError("mode must be one of: sentence, sentence_window, char")

    return [c for c in chunks if len(c) >= int(min_chunk_chars)]


def build_embedding_method_context(
    *,
    provider: str,
    model_name: str,
    chroma_dir: Path = CHROMA_DIR,
    collection_prefix: str = "biblical",
    hf_batch_size: int = 64,
    openai_batch_size: int = 128,
    openai_api_key: Optional[str] = None,
    openai_base_url: Optional[str] = None,
    openai_organization: Optional[str] = None,
    query_prompt: Optional[dict] = None,
    **kwargs,
) -> Dict[str, Any]:
    chromadb = _import_chromadb()
    client = chromadb.PersistentClient(path=str(chroma_dir))

    collection_name = _resolve_collection_name(
        client,
        provider=provider,
        model_name=model_name,
        collection_prefix=collection_prefix,
        chroma_dir=chroma_dir,
        openai_api_key=openai_api_key,
        openai_base_url=openai_base_url,
        openai_organization=openai_organization,
    )
    collection = client.get_collection(collection_name)

    embed_query = _build_query_embedder(
        provider=provider,
        model_name=model_name,
        hf_batch_size=hf_batch_size,
        openai_batch_size=openai_batch_size,
        openai_api_key=openai_api_key,
        openai_base_url=openai_base_url,
        openai_organization=openai_organization,
        query_prompt=query_prompt,
        device=kwargs.get("device", "cpu"),
    )

    return {
        "provider": provider,
        "model_name": model_name,
        "collection_name": collection_name,
        "collection": collection,
        "embed_query": embed_query,
        "inspection_rows": [],
    }

def simple_embedding_method(
    problem_id: str,
    problem_text: str,
    method_context: Dict[str, Any],
    *,
    mode: str = "sentence",
    sentences_per_chunk: int = 2,
    sentence_stride: int = 1,
    char_chunk_size: int = 500,
    char_chunk_overlap: int = 100,
    min_chunk_chars: int = 30,
    top_k: int = 5,
    similarity_threshold: float = 0.45,
) -> List[Dict[str, Any]]:
    chunks = split_into_chunks(
        problem_text,
        mode=mode,
        sentences_per_chunk=sentences_per_chunk,
        sentence_stride=sentence_stride,
        char_chunk_size=char_chunk_size,
        char_chunk_overlap=char_chunk_overlap,
        min_chunk_chars=min_chunk_chars,
    )
    if not chunks:
        return []

    query_embeddings = method_context["embed_query"](chunks)
    query_result = method_context["collection"].query(
        query_embeddings=query_embeddings,
        n_results=max(1, int(top_k)),
        include=["distances", "metadatas"],
    )

    best_scores: Dict[str, float] = {}
    ids_batch = query_result.get("ids", [])
    dists_batch = query_result.get("distances", [])
    metas_batch = query_result.get("metadatas", [])

    for i in range(len(chunks)):
        ids_i = ids_batch[i] if i < len(ids_batch) else []
        dists_i = dists_batch[i] if i < len(dists_batch) else []
        metas_i = metas_batch[i] if i < len(metas_batch) else []

        for j in range(len(ids_i)):
            meta = metas_i[j] if j < len(metas_i) and isinstance(metas_i[j], dict) else {}
            ref = normalize_reference(meta.get("reference", ids_i[j]))
            dist = float(dists_i[j]) if j < len(dists_i) else 1.0
            similarity = 1.0 - dist

            if similarity >= float(similarity_threshold):
                prev = best_scores.get(ref)
                if prev is None or similarity > prev:
                    best_scores[ref] = similarity

    predictions = [
        {
            "problem_id": problem_id,
            "reference": ref,
            "score": score,
            "method": "simple_embedding",
        }
        for ref, score in sorted(best_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    return predictions


In [36]:
def participant_method(problem_id: str, problem_text: str, method_context: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Try to modify the parameters of the naive baseline to improve the results or—implement entirely new method.

    The function applies the method for a single problem. If your approach processed the entire corpus in one go, this function should allow retrieving identified references for any given problem.

    :param problem_id:
    :param problem_text:
    :param method_context: Can be used to pass arbitrary keyword arguments to the method.

    In the baseline method, key parameters to tune are:
      - similarity_threshold: lower = more matches (higher recall), higher = fewer but better matches (higher precision)
      - top_k: number of Bible verse candidates retrieved per text chunk
      - mode: how the problem text is split ("sentence", "sentence_window", "char") as well as related "char_chunk_size", "char_chunk_overlap", "sentences_per_chunk", "sentence_stride".

    See README.md for more details.

    Expected output format per item:
    {
      "problem_id": <str>,
      "reference": <str>,
      "score": <float, optional>,
      "method": <str, optional>
    }
    """
    return simple_embedding_method(
        problem_id,
        problem_text,
        method_context,
        mode="char",
        char_chunk_size=100,
        char_chunk_overlap=50,
        top_k=2,
        similarity_threshold=0.75,
    )


### Running Experiments

- Start with `max_problems=5` for quick iteration, then evaluate all problems.
- If model/collection mismatch occurs, check the mapping file in `data/vectorstores/chroma`.
- Keep the same API contract so your method can be swapped into the evaluator without changes.


In [37]:
model_context = build_embedding_method_context(
    provider="hf",
    model_name="bowphs/LaBerta",
    chroma_dir=CHROMA_DIR,
    device="cpu"
)

# applies participant_method() to the entire dataset
predictions = run_method_on_dataset(
    participant_method,
    problems,
    model_context,
    # max_problems=5,
    show_progress=True,
)

metrics = score_predictions(predictions, ground_truth)
print(pd.Series(metrics))
print(f"Predictions produced: {len(predictions)}")


2026-03-04 13:30:08,591 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: bowphs/LaBerta
2026-03-04 13:30:08,855 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/bowphs/LaBerta/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-03-04 13:30:08,857 - sentence_transformers.SentenceTransformer - WARNING - No sentence-transformers model found with name bowphs/LaBerta. Creating a new one with mean pooling.
2026-03-04 13:30:08,997 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/bowphs/LaBerta/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-03-04 13:30:09,130 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/bowphs/LaBerta/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-04 13:30:09,160 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/bowphs/LaBerta/94fab85783dca8a16529cda2b58760d03bd5d9c1/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: bowphs/LaBerta
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
2026-03-04 13:30:09,409 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/bowphs/LaBerta/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-04 13:30:09,433 - httpx - INFO - HTTP Request: HEAD https://h

Running method:   0%|          | 0/53 [00:00<?, ?problem/s]

true_positives     418.000000
false_positives    122.000000
false_negatives    434.000000
precision            0.774074
recall               0.490610
f1                   0.600575
dtype: float64
Predictions produced: 540


Make adjustments in `method_evaluation` so that the behaviour is as follows.

When the user provides in  `build_embedding_method_context()` a model_name, for which there are no biblical embeddings (i.e., there is not collection in ChromaDB with such model, provider, and prefix 'biblical'), it should create such collection before proceeding with evaluation.